# WC 2026 — Model Accuracy by Tournament Stage

This notebook scores each prediction model against **actual** 2026 World Cup
results, broken down by stage:

| Model | Key |
|-------|-----|
| ELO | `elo` |
| Logistic Regression | `lr_model` |
| Ensemble (LR + LightGBM) | `ensemble_model` |
| Poisson | `poisson_model` |
| Dixon-Coles | `dc_model` |

**Win/draw/loss accuracy**
- *Group stage*: each model's `predict_group_match()` is compared to the real
  score (H / D / A).
- *Knockout rounds*: the full bracket is replayed with actual results frozen
  through the **previous** stage (same logic as `updated_predictions.ipynb`
  with `ACTUALS_THROUGH_STAGE_ID`). The predicted winner is whoever has the
  higher win probability (`predict_knockout_winner`).

**Scoreline accuracy** (Poisson & Dixon-Coles only): exact predicted home/away
goals vs the real score.

Run all cells top-to-bottom. The snapshot section at the end will create any
missing per-stage prediction notebooks under `predictions/snapshots/`.

## 1. Setup

In [ ]:
# --- imports & repo paths ---
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

# Make scripts/ importable when the notebook runs from predictions/
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass

from scripts.wc2026_accuracy import (
    MODELS,
    SCORELINE_MODELS,
    STAGE_ID_TO_NAME,
    AccuracyReport,
    build_team_lookups,
    ensure_snapshots_for_model,
    evaluate_model,
    finished_matches_by_stage,
    format_accuracy_summary,
    load_completed_from_csv,
    load_tournament_tables,
    supplement_from_api,
)

DATA = ROOT / 'data' / 'tournament'
FOOTBALL_DATA_API_KEY = os.environ.get('FOOTBALL_DATA_API_KEY', '')
print(f'Repo root: {ROOT}')

## 2. Load actual results

We start from the committed `data/live/wc2026_results.csv` file, then optionally
merge in any newer FINISHED matches from the football-data.org API.

In [ ]:
matches_df, teams_df, stages_df, results_df = load_tournament_tables()
alloc_df = pd.read_csv(DATA / 'third_place_allocation.csv')
id_to_name, name_to_group, canonical_to_csv = build_team_lookups(teams_df)

# Build lookup of finished matches (frozenset of teams -> scores)
completed = load_completed_from_csv(results_df, matches_df, id_to_name)
completed = supplement_from_api(completed, canonical_to_csv, FOOTBALL_DATA_API_KEY)

# Group finished matches by stage so we only score stages that have played
finished_by_stage = finished_matches_by_stage(results_df, matches_df, id_to_name)

print(f'Total finished matches in CSV : {len(results_df)}')
print(f'Completed lookup entries      : {len(completed)}')
print()
print('Finished matches per stage:')
for sid in sorted(finished_by_stage):
    print(f'  {STAGE_ID_TO_NAME[sid]:<22} {len(finished_by_stage[sid]):>3} matches')

## 3. Evaluate each model

For every model we `%run` its training notebook (same as
`updated_predictions.ipynb`), then call the shared accuracy helpers.

> **Note:** Poisson and Dixon-Coles refit on the master dataset each time, so
> this cell can take several minutes.

In [ ]:
def load_model_notebook(model_key: str) -> None:
    """Run models/<model_key>.ipynb to expose predict_* functions in the kernel."""
    path = ROOT / 'models' / f'{model_key}.ipynb'
    if not path.exists():
        raise FileNotFoundError(path)
    # %run is the same mechanism updated_predictions.ipynb uses
    get_ipython().run_line_magic('run', str(path))


all_reports: list[AccuracyReport] = []

for model_key, model_label in MODELS.items():
    print('=' * 60)
    print(f'Loading model: {model_label} ({model_key})')
    print('=' * 60)
    load_model_notebook(model_key)

    # After %run, the model notebook has defined these functions in the kernel
    report = evaluate_model(
        model_key=model_key,
        matches_df=matches_df,
        alloc_df=alloc_df,
        id_to_name=id_to_name,
        name_to_group=name_to_group,
        completed=completed,
        finished_by_stage=finished_by_stage,
        predict_group=predict_group_match,
        predict_knockout=predict_knockout_winner,
        get_elo=get_elo,
    )
    all_reports.append(report)
    print(f'  Scored {len(report.stages)} stage(s)\n')

## 4. Accuracy summary

Printed in the format you requested — win/draw/loss accuracy per stage, plus
exact scoreline accuracy for Poisson and Dixon-Coles.

In [ ]:
summary_text = format_accuracy_summary(all_reports)
print(summary_text)

In [ ]:
# --- tabular view (handy for copying into a spreadsheet) ---
rows = []
for report in all_reports:
    for s in report.stages:
        rows.append({
            'Stage': s.stage_name,
            'Model': s.model_label,
            'Win/D/L Accuracy %': round(s.win_accuracy, 1),
            'Correct': s.n_correct,
            'Played': s.n_matches,
            'Score Accuracy %': round(s.score_accuracy, 1) if s.score_accuracy is not None else None,
            'Exact Scores': s.n_score_correct if s.score_accuracy is not None else None,
        })

accuracy_df = pd.DataFrame(rows)
display(accuracy_df.pivot_table(
    index='Stage',
    columns='Model',
    values='Win/D/L Accuracy %',
    aggfunc='first',
).reindex(columns=[MODELS[k] for k in MODELS]))

## 5. Per-stage detail tables

In [ ]:
for stage_id in sorted({s.stage_id for r in all_reports for s in r.stages}):
    stage_name = STAGE_ID_TO_NAME[stage_id]
    display(Markdown(f'### {stage_name}'))

    win_rows = []
    score_rows = []
    for report in all_reports:
        match = next((s for s in report.stages if s.stage_id == stage_id), None)
        if match is None:
            continue
        win_rows.append({
            'Model': match.model_label,
            'Accuracy': f"{match.win_accuracy:.0f}%",
            'Correct': match.n_correct,
            'Played': match.n_matches,
        })
        if match.score_accuracy is not None:
            score_rows.append({
                'Model': match.model_label,
                'Accuracy': f"{match.score_accuracy:.0f}%",
                'Exact': match.n_score_correct,
                'Played': match.n_score_matches,
            })

    print(f'{stage_name} Win Prediction Accuracy:')
    display(pd.DataFrame(win_rows))

    if score_rows:
        print(f'{stage_name} Score Prediction Accuracy:')
        display(pd.DataFrame(score_rows))

---
## 6. Snapshot section

Permanent prediction snapshots freeze what each model predicted **right after**
a stage ended — before later results overwrite the view in
`updated_predictions.ipynb`.

For each model and each **completed** stage, this section creates a snapshot
notebook under `predictions/snapshots/` **only if one does not already exist**.

Naming convention (from `scripts/make_stage_snapshot.py`):
- Poisson (default): `after_group_stage.ipynb`, `after_round_of_32.ipynb`, …
- Other models: `after_group_stage_elo.ipynb`, `after_round_of_32_dc_model.ipynb`, …

> Snapshots re-execute `updated_predictions.ipynb` with `ACTUALS_THROUGH_STAGE_ID`
> set, so each one can take a few minutes. Already-existing files are skipped.

In [ ]:
from scripts.make_stage_snapshot import STAGES, snapshot_path, stage_is_complete
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
print('Snapshot readiness (completed stage + file status):\n')
for model_key, model_label in MODELS.items():
    print(f'--- {model_label} ---')
    for stage in STAGES:
        complete = stage_is_complete(matches_df, stage.stage_id, now)
        path = snapshot_path(stage, model_key)
        status = 'exists' if path.exists() else ('will create' if complete else 'stage not finished')
        print(f'  [{stage.stage_id}] after {stage.completed_label:<13} -> {stage.preview_label:<13} : {status}')
    print()

In [ ]:
# Generate any missing snapshots (skip existing files)
created_all: list[str] = []

for model_key, model_label in MODELS.items():
    print(f'Checking snapshots for {model_label}...')
    created = ensure_snapshots_for_model(model_key, matches_df, force=False)
    if created:
        for p in created:
            print(f'  CREATED: {p}')
        created_all.extend(created)
    else:
        print('  Nothing new to create.')

print()
if created_all:
    print(f'Generated {len(created_all)} new snapshot notebook(s).')
else:
    print('All available snapshots already exist (or no stage has finished yet).')

### Snapshot index

Quick list of every snapshot file currently on disk.

In [ ]:
snapshot_dir = ROOT / 'predictions' / 'snapshots'
snap_files = sorted(snapshot_dir.glob('after_*.ipynb'))
if snap_files:
    for f in snap_files:
        html = f.with_suffix('.html')
        html_note = ' (+ .html)' if html.exists() else ''
        print(f'{f.name}{html_note}')
else:
    print('No snapshot notebooks found yet.')